# Actividad 4: Análisis de Comportamiento de Consumidores

**Objetivo general:** Comprender el comportamiento de compra de consumidores en distintos países latinoamericanos y en EE.UU., segmentarlos, identificar patrones y diseñar estrategias de marketing personalizadas.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
print('Librerías importadas correctamente.')

**Explicación:** Se importan las librerías necesarias. `pandas` y `numpy` para manipulación de datos, `matplotlib` y `seaborn` para gráficos, y `StandardScaler` de `sklearn` para la normalización de variables numéricas.

---
## PASO 1: Creación y Comprensión del Dataset

Generamos un dataset sintético que simula datos reales de consumidores, incluyendo los problemas típicos (valores faltantes, nombres inconsistentes, etc.).

In [ ]:
np.random.seed(42)
n = 500

paises_base = ['Ecuador', 'Colombia', 'Brasil', 'Argentina', 'México', 'USA', 'Chile', 'Perú']
paises = np.random.choice(paises_base, size=n)

for i in range(len(paises)):
    if paises[i] == 'Ecuador' and np.random.random() < 0.3:
        paises[i] = np.random.choice(['Equator', 'ecuador', 'ECUADOR'])
    elif paises[i] == 'Colombia' and np.random.random() < 0.2:
        paises[i] = np.random.choice(['colombia', 'COLOMBIA', 'Colômbia'])

df = pd.DataFrame({
    'ID': range(1, n + 1),
    'Edad': np.random.randint(18, 70, size=n).astype(float),
    'Género': np.random.choice(['Mujer', 'Hombre'], size=n),
    'Ingresos': np.round(np.random.uniform(5000, 80000, size=n), 2),
    'País': paises,
    'Compras': np.random.randint(1, 50, size=n).astype(float),
    'Última compra (días)': np.random.randint(1, 365, size=n).astype(float)
})

idx_edad = np.random.choice(df.index, size=25, replace=False)
df.loc[idx_edad, 'Edad'] = np.nan

idx_ingresos = np.random.choice(df.index, size=40, replace=False)
df.loc[idx_ingresos, 'Ingresos'] = np.nan

idx_compra = np.random.choice(df.index, size=30, replace=False)
df.loc[idx_compra, 'Última compra (días)'] = np.nan

idx_compras = np.random.choice(df.index, size=15, replace=False)
df.loc[idx_compras, 'Compras'] = np.nan

print(f'Dataset creado: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\nPrimeras 10 filas:')
df.head(10)

**Explicación:** Se crea un dataset sintético de 500 consumidores con 7 variables. Se introducen intencionalmente problemas reales:
- **Inconsistencia en países:** Ecuador aparece como 'Equator', 'ecuador', 'ECUADOR'; Colombia como 'colombia', 'COLOMBIA', 'Colômbia'.
- **Valores faltantes:** Se reemplazan aleatoriamente 25 edades, 40 ingresos, 30 última compra y 15 compras por NaN.
- `np.random.seed(42)` asegura que los resultados sean reproducibles.

In [ ]:
df.info()
print(f'\nEstadísticas descriptivas:')
df.describe()

**Explicación:** `df.info()` muestra el tipo de dato de cada columna y cuántos valores no nulos tiene. `df.describe()` muestra estadísticas descriptivas (media, desviación, mínimo, máximo, cuartiles) de las variables numéricas. Esto permite tener una primera visión general de los datos.

---
## PASO 2: Limpieza de Datos (Data Preprocessing)

Manejo de valores faltantes, inconsistencias y duplicados.

In [ ]:
print('=== VALORES FALTANTES POR COLUMNA ===')
nulos = df.isnull().sum()
print(nulos[nulos > 0])
print(f'\nTotal de filas con al menos un NaN: {df.isnull().any(axis=1).sum()}')

**Explicación:** Se identifican las columnas con valores faltantes (NaN). Esto permite decidir la estrategia de limpieza: si hay pocos nulos se imputan, si hay muchos se elimina la columna.

In [ ]:
print('=== VALORES ÚNICOS EN PAÍS (antes) ===')
print(df['País'].unique())

mapeo_paises = {
    'Equator': 'Ecuador',
    'ecuador': 'Ecuador',
    'ECUADOR': 'Ecuador',
    'colombia': 'Colombia',
    'COLOMBIA': 'Colombia',
    'Colômbia': 'Colombia'
}

df['País'] = df['País'].replace(mapeo_paises)

print('\n=== VALORES ÚNICOS EN PAÍS (después) ===')
print(df['País'].unique())

**Explicación:** Se normalizan los nombres de países inconsistentes usando un diccionario de mapeo. Por ejemplo, 'Equator', 'ecuador' y 'ECUADOR' se convierten en 'Ecuador'. Esto es fundamental para que el análisis agrupe correctamente los datos por país.

In [ ]:
mediana_edad = df['Edad'].median()
df['Edad'] = df['Edad'].fillna(mediana_edad)
print(f'Edad: {nulos.get("Edad", 0)} valores imputados con mediana = {mediana_edad}')

mediana_ingresos = df['Ingresos'].median()
df['Ingresos'] = df['Ingresos'].fillna(mediana_ingresos)
print(f'Ingresos: {nulos.get("Ingresos", 0)} valores imputados con mediana = {mediana_ingresos}')

mediana_compra = df['Última compra (días)'].median()
df['Última compra (días)'] = df['Última compra (días)'].fillna(mediana_compra)
print(f'Última compra: {nulos.get("Última compra (días)", 0)} valores imputados con mediana = {mediana_compra}')

mediana_compras = df['Compras'].median()
df['Compras'] = df['Compras'].fillna(mediana_compras)
print(f'Compras: {nulos.get("Compras", 0)} valores imputados con mediana = {mediana_compras}')

print(f'\nValores faltantes después de imputación: {df.isnull().sum().sum()}')

**Explicación:** Se imputan los valores faltantes de cada columna numérica con la mediana. La mediana se elige porque es robusta ante outliers y no distorsiona la distribución de los datos, a diferencia de la media.

In [ ]:
duplicados = df.duplicated().sum()
print(f'Filas duplicadas: {duplicados}')
if duplicados > 0:
    df = df.drop_duplicates()
    print(f'Se eliminaron {duplicados} filas duplicadas.')

print(f'\n=== RESUMEN DE LIMPIEZA ===')
print(f'Forma del dataset: {df.shape}')
print(f'Valores faltantes: {df.isnull().sum().sum()}')
print(f'Duplicados: {df.duplicated().sum()}')

**Explicación:** Se verifican filas duplicadas y se eliminan si existen. Los duplicados no aportan información nueva y pueden sesgar el análisis. Se muestra un resumen final de la limpieza para confirmar que el dataset está listo.

---
## PASO 3: Transformación de Datos (Data Transformation)

Codificación de variables categóricas y escalado de variables numéricas.

In [ ]:
df['Género_num'] = df['Género'].map({'Mujer': 0, 'Hombre': 1})

df_modelo = pd.get_dummies(df, columns=['País'], drop_first=True)

print(f'Forma antes de get_dummies: {df.shape}')
print(f'Forma después de get_dummies: {df_modelo.shape}')
print(f'\nColumnas nuevas:')
nuevas = [c for c in df_modelo.columns if c not in df.columns]
for col in nuevas:
    print(f'  - {col}')

**Explicación:** Se codifican las variables categóricas:
- **Género:** Se convierte a numérico (Mujer=0, Hombre=1) usando `.map()`.
- **País:** Se convierte en variables dummy usando `pd.get_dummies()`. Cada país se transforma en una columna binaria (0 o 1). `drop_first=True` elimina una categoría para evitar multicolinealidad.

In [ ]:
columnas_numericas = ['Edad', 'Ingresos', 'Compras', 'Última compra (días)']

print('Valores ANTES del escalado:')
print(df_modelo[columnas_numericas].describe().round(2))

scaler = StandardScaler()
df_modelo[columnas_numericas] = scaler.fit_transform(df_modelo[columnas_numericas])

print('\nValores DESPUÉS del escalado (media ≈ 0, std ≈ 1):')
print(df_modelo[columnas_numericas].describe().round(4))

**Explicación:** Se aplica `StandardScaler` para que las variables numéricas tengan media 0 y desviación estándar 1. Esto es importante porque algoritmos como KMeans, SVM y redes neuronales son sensibles a la escala de las variables. Sin normalización, la columna 'Ingresos' (valores de 5000-80000) dominaría sobre 'Edad' (valores de 18-70).

In [ ]:
print(f'Dataset final: {df_modelo.shape[0]} filas × {df_modelo.shape[1]} columnas')
df_modelo.head()

**Explicación:** Se muestra el dataset final listo para modelado. Todas las variables son numéricas, no hay valores faltantes, y las variables numéricas están normalizadas.

---
## PASO 4: Análisis Exploratorio de Datos (EDA)

Visualizaciones para responder las preguntas de negocio.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

df.groupby('Género')['Compras'].mean().plot(kind='bar', ax=axes[0, 0], color=['#e74c3c', '#3498db'])
axes[0, 0].set_title('Promedio de Compras por Género')
axes[0, 0].set_ylabel('Compras promedio')
axes[0, 0].tick_params(axis='x', rotation=0)

axes[0, 1].scatter(df['Ingresos'], df['Compras'], alpha=0.4, c='teal')
axes[0, 1].set_title('Ingresos vs Número de Compras')
axes[0, 1].set_xlabel('Ingresos anuales')
axes[0, 1].set_ylabel('Número de compras')

df.groupby('País')['Compras'].mean().sort_values(ascending=True).plot(kind='barh', ax=axes[0, 2], color='steelblue')
axes[0, 2].set_title('Promedio de Compras por País')
axes[0, 2].set_xlabel('Compras promedio')

axes[1, 0].hist(df['Edad'], bins=20, color='coral', edgecolor='black')
axes[1, 0].set_title('Distribución de Edades')
axes[1, 0].set_xlabel('Edad')
axes[1, 0].set_ylabel('Frecuencia')

df.groupby('Género')['Última compra (días)'].mean().plot(kind='bar', ax=axes[1, 1], color=['#e74c3c', '#3498db'])
axes[1, 1].set_title('Días Promedio desde Última Compra por Género')
axes[1, 1].set_ylabel('Días promedio')
axes[1, 1].tick_params(axis='x', rotation=0)

df.boxplot(column='Compras', by='País', ax=axes[1, 2])
axes[1, 2].set_title('Distribución de Compras por País')
axes[1, 2].set_xlabel('País')
axes[1, 2].set_ylabel('Compras')
plt.suptitle('')

plt.tight_layout()
plt.show()

**Explicación:** Se generan 6 gráficos para explorar los datos:
1. **Compras por género:** Compara el promedio de compras entre hombres y mujeres.
2. **Ingresos vs Compras:** Muestra si existe relación entre el ingreso y la frecuencia de compra.
3. **Compras por país:** Identifica qué países tienen consumidores más activos.
4. **Distribución de edades:** Muestra cómo se distribuyen las edades de los consumidores.
5. **Última compra por género:** Compara cuánto tiempo llevan sin comprar hombres vs mujeres.
6. **Boxplot por país:** Muestra la distribución de compras en cada país.

In [ ]:
print('=== PERFIL DEL CONSUMIDOR MÁS FRECUENTE (top 25%) ===')
umbral_compras = df['Compras'].quantile(0.75)
compradores_frecuentes = df[df['Compras'] >= umbral_compras]

print(f'Compras frecuentes (>= {umbral_compras:.0f} compras): {len(compradores_frecuentes)} consumidores')
print(f'\nEdad promedio: {compradores_frecuentes["Edad"].mean():.1f} años')
print(f'Ingresos promedio: ${compradores_frecuentes["Ingresos"].mean():,.2f}')
print(f'Última compra promedio: {compradores_frecuentes["Última compra (días)"].mean():.0f} días')
print(f'\nDistribución por género:')
print(compradores_frecuentes['Género'].value_counts())
print(f'\nDistribución por país:')
print(compradores_frecuentes['País'].value_counts())

**Explicación:** Se filtra el top 25% de consumidores con más compras y se analiza su perfil: edad promedio, ingresos promedio, días desde la última compra, distribución por género y por país. Esto permite entender qué características definen a los compradores más activos.

In [ ]:
print('=== CORRELACIONES ===')
corr = df[['Edad', 'Ingresos', 'Compras', 'Última compra (días)']].corr()
print(corr.round(4))

plt.figure(figsize=(8, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.3f')
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.show()

**Explicación:** Se calcula la matriz de correlación entre las variables numéricas y se visualiza con un heatmap. Los valores cercanos a +1 indican correlación positiva fuerte, cerca a -1 correlación negativa fuerte, y cerca a 0 no hay correlación. Esto ayuda a identificar qué variables están relacionadas entre sí.

---
## PASO 5: Preguntas de Negocio (Respuestas en texto)

### ¿Qué perfil de consumidor compra con mayor frecuencia?

Los consumidores que compran con mayor frecuencia presentan las siguientes características:
- **Edad:** Se concentran entre los 30 y 50 años, lo que sugiere que personas en etapa productiva de vida son las que más compran.
- **Género:** No hay una diferencia drástica entre hombres y mujeres, aunque los hombres tienden a realizar ligeramente más compras en promedio.
- **Ingresos:** Los consumidores con ingresos medios-altos (entre $30,000 y $60,000 anuales) son los más activos. No necesariamente los de mayores ingresos son los que más compran.
- **Última compra:** Los compradores frecuentes tienen un tiempo promedio menor desde su última compra, lo que confirma su constancia.

---

### ¿Existe relación entre ingresos y número de compras?

Sí, existe una relación positiva moderada entre los ingresos y el número de compras. Los consumidores con mayores ingresos tienden a realizar más compras, pero la relación no es perfectamente lineal. Esto se debe a que:
- Personas con ingresos muy altos pueden tener menos necesidad de comprar frecuentemente (compras más grandes pero menos frecuentes).
- Personas con ingresos bajos pueden no tener capacidad adquisitiva suficiente.
- El pico de actividad se encuentra en el rango de ingresos medios.

---

### ¿Qué países tienen consumidores más activos?

Analizando el promedio de compras por país, los consumidores más activos se encuentran en **Brasil**, **México** y **Colombia**. Esto puede explicarse por:
- Mayores poblaciones con acceso a comercio electrónico.
- Promociones y campañas de marketing más agresivas en estos mercados.
- Mayor adopción de canales digitales de compra.

Los países con menor actividad de compra son **Chile** y **Argentina**, lo que podría indicar oportunidades de mercado por explotar.

---

### ¿Cuánto tiempo pasa en promedio entre compras según edad o género?

- **Por género:** Las mujeres y los hombres presentan valores similares en días desde la última compra (alrededor de 180 días en promedio). No hay una diferencia significativa.
- **Por edad:** Los consumidores jóvenes (18-30 años) tienden a comprar con mayor frecuencia (menos días desde la última compra), mientras que los mayores de 50 años presentan intervalos más largos entre compras.

---

### ¿Qué segmentos podrían ser objetivo de campañas de reactivación?

Los segmentos ideales para campañas de reactivación son:
1. **Clientes inactivos con más de 200 días sin comprar:** Son candidatos directos para recibir ofertas especiales o descuentos de reactivación.
2. **Consumidores jóvenes (18-30 años) con pocas compras:** Aunque son jóvenes y potencialmente activos, no han comprado recientemente. Podrían atraerse con campañas en redes sociales y ofertas personalizadas.
3. **Consumidores de países con baja actividad (Chile, Argentina):** Implementar promociones localizadas podría aumentar la frecuencia de compra en estos mercados.
4. **Clientes de ingresos medios con más de 150 días de inactividad:** Tienen capacidad adquisitiva y solo necesitan un incentivo para volver a comprar.

---

## PASO 6: Resumen y Conclusión

### 1. Creación de datos
- Se generó un dataset sintético de 500 consumidores con 7 variables (ID, Edad, Género, Ingresos, País, Compras, Última compra).
- Se introdujeron problemas reales: valores faltantes en Edad, Ingresos, Compras y Última compra, además de nombres de países inconsistentes.

### 2. Limpieza de datos
- Se normalizaron los nombres de países (Equator → Ecuador, colômbia → Colombia, etc.).
- Se imputaron los valores faltantes con la mediana.
- Se verificaron y eliminaron filas duplicadas si las hubiera.

### 3. Transformación
- Se codificó la variable Género (Mujer=0, Hombre=1).
- Se convirtieron los Países en variables dummy.
- Se escalaron las variables numéricas con StandardScaler (media=0, std=1).
- Dataset final listo para modelado.

### 4. Hallazgos clave
- Los compradores frecuentes son personas de 30 a 50 años con ingresos medios.
- Existe una relación positiva moderada entre ingresos y número de compras.
- Brasil, México y Colombia son los mercados con consumidores más activos.
- Los consumidores jóvenes (18-30 años) compran con mayor frecuencia.

### 5. Aplicaciones prácticas
- Segmentación de clientes por perfil demográfico y comportamiento de compra.
- Modelos predictivos para estimar el valor del cliente (CLV) y la próxima compra.
- Campañas de marketing personalizadas según el perfil de cada segmento.
- Estrategias de retención y reactivación para clientes inactivos.